# Phase 5: full-fleet extract + build (`ami_raw` / `ami_meter`)

Produces the actual deliverable: a full year of `ami_raw` (ground truth --
`pv_generation`, `gross_load`) and `ami_meter` (the synthetic smart meter --
per-phase `ac_load_net` readings), for every `CLEAN_SITE_IDS` site that
survives resolution, written to the local Parquet store
(`ami_config.STORE_DIR` -- outside OneDrive, see that module's own
docstring on why).

Four steps, each building on the last, none re-litigated here:

1. **Resolve fleet-wide**, on one representative day (same duplicate/
   inactive/power-correction logic as `04_site_resolution.ipynb`, just at
   full `CLEAN_SITE_IDS` scale instead of a validation sample), then drop
   flagged and manual-review sites entirely (most conservative choice,
   agreed 2026-08-27).
2. **Extract** a full year for the survivors only -- never for a site
   we're about to drop -- chunked to local Parquet (`lib/ami_extract.py`).
3. **Revalidate** against the full landed year, not just the one sampled
   day: a circuit that goes inactive mid-year is dropped on its own; a
   site that fails the reconstruction/storage check in any month is
   excluded entirely (`lib/ami_revalidate.py`).
4. **Build** `ami_raw`/`ami_meter` from the fully revalidated set, one
   landed month at a time (`lib/ami_build.py`).

This notebook orchestrates only -- all four steps' actual logic lives in
`lib/ami_*.py`, unit-tested and dry-run-verified before this notebook was
written.


In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

_current = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (_current, *_current.parents) if (p / "bms_sa_review").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError(f"Could not locate the CICCADA repository root from {_current}")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

from bms_sa_review.ami_data_analysis.config import ami_config as Config
from bms_sa_review.ami_data_analysis.lib import ami_athena as Athena
from bms_sa_review.ami_data_analysis.lib import ami_taxonomy as Taxonomy
from bms_sa_review.ami_data_analysis.lib import ami_resolution as Resolution
from bms_sa_review.ami_data_analysis.lib import ami_signal as Signal
from bms_sa_review.ami_data_analysis.lib import ami_plots as Plots
from bms_sa_review.ami_data_analysis.lib import ami_extract as Extract
from bms_sa_review.ami_data_analysis.lib import ami_revalidate as Reval
from bms_sa_review.ami_data_analysis.lib import ami_build as Build

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

Athena.reset_scan_log()
Athena.require_credentials()
print("Credentials OK. Starting Phase 5 extract + build.")


Credentials OK. Starting Phase 5 extract + build.


## 1. Fleet-wide resolution

### 1a. Cohort: recompute `CLEAN_SITE_IDS` fresh

Identical to `04_site_resolution.ipynb`'s own section 1 (same queries, same
restriction to sites with at least one PV-side AND one load-side
circuit_type present) -- kept self-contained here rather than depending on
that notebook having been run in the same kernel session.


In [2]:
raw_circuit_type_counts = Athena.aq(
    """
    SELECT circuit_type, is_pv, count(*) AS n_circuits, count(DISTINCT site_id) AS n_sites
    FROM meta_up23c
    GROUP BY circuit_type, is_pv
    """,
    database=Config.SAI, label="circuit_type census (Phase 5)",
)
census = Taxonomy.summarise_circuit_types(raw_circuit_type_counts)
census = Taxonomy.flag_suspected_aggregates(census)
suspects = census[census.suspected_aggregate]

load_candidate_type = "ac_load_net"
pv_candidate_type = "pv_site_net"
assert load_candidate_type in suspects.circuit_type.tolist(), (
    f"{load_candidate_type!r} not flagged as a suspected aggregate this run -- "
    "the fleet's circuit_type census may have changed; re-check before proceeding."
)
assert pv_candidate_type in suspects.circuit_type.tolist(), (
    f"{pv_candidate_type!r} not flagged as a suspected aggregate this run -- "
    "the fleet's circuit_type census may have changed; re-check before proceeding."
)
print(f"Load-side candidate: {load_candidate_type!r}. PV-side candidate: {pv_candidate_type!r}.")


Load-side candidate: 'ac_load_net'. PV-side candidate: 'pv_site_net'.


In [3]:
meta_local = Athena.aq(
    """
    SELECT circuit_id, site_id, circuit_type, is_pv, circuit_polarity, ac_capacity_kw,
           dc_capacity_kw, export_limit_kw, inverter_count,
           device_id, m_id, device_type, voltage_class, min_time, max_time, s_99,
           state, postcode, dnsp_name, flex_export_detected, manufacturer, model,
           monitoring_start, pv_install_date
    FROM meta_up23c
    """,
    database=Config.SAI, label="meta_up23c full projection (Phase 5)",
)
print(f"meta_local: {len(meta_local):,} rows, {meta_local.circuit_id.nunique():,} distinct circuits")

cohort = Taxonomy.cohort_completeness(meta_local)
pv_types = census[census.is_pv.astype(bool)].circuit_type.tolist()
load_types = census[~census.is_pv.astype(bool)].circuit_type.tolist()

pv_cols = [t for t in pv_types if t in cohort.columns]
load_cols = [t for t in load_types if t in cohort.columns]
pv_present = cohort[pv_cols].sum(axis=1) > 0 if pv_cols else pd.Series(False, index=cohort.index)
load_present = cohort[load_cols].sum(axis=1) > 0 if load_cols else pd.Series(False, index=cohort.index)
cohort_with_both = cohort[pv_present & load_present].reset_index(drop=True)
print(f"{len(cohort_with_both):,} of {len(cohort):,} sites have at least one PV-side AND "
      "one load-side circuit_type present.")

pv_cohort = cohort[pv_present].reset_index(drop=True)


meta_local: 60,570 rows, 60,570 distinct circuits
15,167 of 16,148 sites have at least one PV-side AND one load-side circuit_type present.


In [4]:
CLEAN_AC_LOAD_NET_COUNTS = (1, 3)

counts_by_site = pv_cohort[["site_id", load_candidate_type]].rename(
    columns={load_candidate_type: "n_circuits"}
)
counts_by_site = counts_by_site[counts_by_site.n_circuits > 0]

CLEAN_SITE_IDS, OTHER_COUNT_SITE_IDS = Resolution.classify_circuit_counts(
    counts_by_site, clean_counts=CLEAN_AC_LOAD_NET_COUNTS
)
print(f"{len(CLEAN_SITE_IDS):,} CLEAN_SITE_IDS (1 or 3 `{load_candidate_type}` circuits).")
print(f"{len(OTHER_COUNT_SITE_IDS):,} OTHER_COUNT_SITE_IDS -- DEFERRED to a later phase, "
      "not built here (agreed 2026-08-27).")


14,353 CLEAN_SITE_IDS (1 or 3 `ac_load_net` circuits).
695 OTHER_COUNT_SITE_IDS -- DEFERRED to a later phase, not built here (agreed 2026-08-27).


### 1a-2. Trial scope -- EDIT THESE TWO before a full-fleet/full-year run

Defaults below run the FULL `CLEAN_SITE_IDS` cohort and the FULL year --
i.e. the real, expensive production run. For a first real trial (checking
actual Athena cost/timing/file sizes before committing to ~11,000 sites x
12 months), set `TRIAL_N_SITES` to a small number and/or `TRIAL_MONTHS` to
a short range. Everything downstream (extraction, revalidation, build)
automatically follows whatever `RESOLUTION_SITE_IDS`/`TRIAL_MONTHS`
resolve to here -- no other cell needs editing for a trial run.


In [5]:
TRIAL_N_SITES = None   # e.g. 50 for a small trial; None = the full CLEAN_SITE_IDS cohort
TRIAL_MONTHS = (2025, 1, 2025, 12)   # (start_year, start_month, end_year, end_month); e.g. (2025, 6, 2025, 6) for one month

if TRIAL_N_SITES is not None:
    site_pool = pd.Series(CLEAN_SITE_IDS)
    RESOLUTION_SITE_IDS = sorted(
        site_pool.sample(n=min(TRIAL_N_SITES, len(site_pool)), random_state=0).tolist()
    )
else:
    RESOLUTION_SITE_IDS = CLEAN_SITE_IDS

print(f"Running Phase 5 over {len(RESOLUTION_SITE_IDS):,} of {len(CLEAN_SITE_IDS):,} "
      f"CLEAN_SITE_IDS sites, {TRIAL_MONTHS[0]}-{TRIAL_MONTHS[1]:02d} through "
      f"{TRIAL_MONTHS[2]}-{TRIAL_MONTHS[3]:02d}.")


Running Phase 5 over 14,353 of 14,353 CLEAN_SITE_IDS sites, 2025-01 through 2025-12.


### 1b. Pull ONE representative day, for ALL `CLEAN_SITE_IDS`

Same day Phase 3's fleet scan and Phase 4's validation batch already used
(2025-06-01, AEST), so this run's day-level findings stay comparable to
both. Unlike Phase 4's 200-site sample, this is the FULL `CLEAN_SITE_IDS`
cohort -- tens of thousands of circuits -- so the circuit-id list itself is
chunked (`ami_extract.chunk_circuit_ids`) the same way the full-year
extract will be, not just split by `is_pv`. Still one day, so the result
comfortably fits in memory as a single DataFrame (Phase 4's resolution
logic is designed to run on exactly this shape).


In [6]:
FLEET_YEAR, FLEET_MONTH = 2025, 6
FLEET_DAY_START = "2025-06-01 00:00:00"
FLEET_DAY_END = "2025-06-02 00:00:00"

KEPT_TYPES = [load_candidate_type, pv_candidate_type]
fleet_meta = meta_local[
    meta_local.site_id.isin(RESOLUTION_SITE_IDS) & meta_local.circuit_type.isin(KEPT_TYPES)
]
print(f"{fleet_meta.circuit_id.nunique():,} candidate circuits across "
      f"{fleet_meta.site_id.nunique():,} sites in this run's cohort.")

fleet_pulls = []
for is_pv_value in (False, True):
    side_ids = fleet_meta.loc[fleet_meta.is_pv == is_pv_value, "circuit_id"].tolist()
    for id_chunk in Extract.chunk_circuit_ids(side_ids, chunk_size=800):
        sql = Extract.build_extract_sql(id_chunk, FLEET_YEAR, FLEET_MONTH, is_pv_value)
        sql += f"""
  AND t_stamp >= TIMESTAMP '{FLEET_DAY_START}'
  AND t_stamp <  TIMESTAMP '{FLEET_DAY_END}'"""
        pulled = Athena.aq(
            sql, database=Config.SAI,
            label=f"fleet day sample {FLEET_YEAR}-{FLEET_MONTH:02d} is_pv={is_pv_value} "
                  f"({len(id_chunk)} circuits)",
        )
        fleet_pulls.append(pulled)

fleet_sample = pd.concat(fleet_pulls, ignore_index=True) if fleet_pulls else pd.DataFrame()
if len(fleet_sample):
    # find_duplicate_circuits/find_inactive_circuits need site_id (and the
    # resolution/interval-table steps need circuit_type/device_id too) --
    # the `ts` pull itself never carries these, only `circuit_id` (mirrors
    # notebook 04's identical merge-after-pull pattern).
    fleet_sample = fleet_sample.merge(
        fleet_meta[["circuit_id", "site_id", "circuit_type", "device_id", "circuit_polarity"]],
        on="circuit_id", how="left",
    )
print(f"fleet_sample: {len(fleet_sample):,} rows, {fleet_sample.circuit_id.nunique():,} circuits.")
Athena.scan_report()


47,559 candidate circuits across 14,353 sites in this run's cohort.
fleet_sample: 8,803,475 rows, 30,951 circuits.
62 queries, 342.22 GB scanned, ~AUD 2.6737 (billed at a 10.00 MB minimum per query)


,label,database,n_rows,seconds,scanned,scanned_bytes,cost,source
0,circuit_type census (Phase 5),solar_analytics_iceberg,33,3.18,1.14 MB,1.194973e+06,0.0001,query_metadata
1,meta_up23c full projection (Phase 5),solar_analytics_iceberg,60570,5.18,7.58 MB,7.944777e+06,0.0001,query_metadata
2,fleet day sample 2025-06 is_pv=False (800 circ...,solar_analytics_iceberg,9210,5.45,2.62 GB,2.809043e+09,0.0204,query_metadata
3,fleet day sample 2025-06 is_pv=False (800 circ...,solar_analytics_iceberg,63273,4.63,5.04 GB,5.416319e+09,0.0394,query_metadata
4,fleet day sample 2025-06 is_pv=False (800 circ...,solar_analytics_iceberg,77199,4.95,5.61 GB,6.020076e+09,0.0438,query_metadata
...,...,...,...,...,...,...,...,...
57,fleet day sample 2025-06 is_pv=True (800 circu...,solar_analytics_iceberg,164093,7.36,5.57 GB,5.981379e+09,0.0435,query_metadata
58,fleet day sample 2025-06 is_pv=True (800 circu...,solar_analytics_iceberg,175370,7.24,5.66 GB,6.072626e+09,0.0442,query_metadata
59,fleet day sample 2025-06 is_pv=True (800 circu...,solar_analytics_iceberg,199188,6.28,5.58 GB,5.991848e+09,0.0436,query_metadata
60,fleet day sample 2025-06 is_pv=True (800 circu...,solar_analytics_iceberg,177799,7.20,4.92 GB,5.288103e+09,0.0385,query_metadata


### 1c. Resolve every site's circuits, fleet-wide


In [7]:
fleet_resolution = Resolution.resolve_site_circuits(
    fleet_meta, fleet_sample,
    load_type=load_candidate_type, pv_type=pv_candidate_type,
    nominal_interval_minutes=Config.SOURCE_INTERVAL_MINUTES,
)
n_dropped = int((~fleet_resolution.kept).sum())
print(f"{len(fleet_resolution):,} candidate circuit(s) across "
      f"{fleet_resolution.site_id.nunique():,} sites -- {n_dropped:,} dropped "
      "(duplicate or inactive) before any storage/reconstruction check.")
display(fleet_resolution.head(20))


47,559 candidate circuit(s) across 14,353 sites -- 7,212 dropped (duplicate or inactive) before any storage/reconstruction check.


,site_id,circuit_id,circuit_type,device_id,is_pv,kept,drop_reason,needs_manual_review,power_correction_applied,implied_interval_minutes
0,1233585204,219150,pv_site_net,124179,True,True,None,False,False,NaN
1,158521185,239893,pv_site_net,121143,True,True,None,False,False,5.000000
2,248529252,276769,ac_load_net,127872,False,True,None,False,False,5.000009
3,248529252,276768,ac_load_net,127872,False,True,None,False,False,5.000005
4,804338564,298418,pv_site_net,132056,True,True,None,False,False,5.000000
5,1706869440,308033,pv_site_net,121381,True,True,None,True,False,5.000000
6,1706869440,308039,ac_load_net,172833,False,True,None,False,False,5.000000
7,656141920,329989,ac_load_net,111021,False,True,None,False,False,5.000000
8,656141920,329988,pv_site_net,111021,True,False,duplicate_same_type,True,False,NaN
9,656141920,329987,pv_site_net,111021,True,False,duplicate_same_type,True,False,NaN


In [8]:
fleet_resolution[fleet_resolution["site_id"] == 360624571]

,site_id,circuit_id,circuit_type,device_id,is_pv,kept,drop_reason,needs_manual_review,power_correction_applied,implied_interval_minutes
33771,360624571,631836,ac_load_net,112534,False,True,None,False,True,4.895823
34577,360624571,631838,pv_site_net,112534,True,True,None,False,True,4.890260


### 1d. Load reconstruction & storage sanity check, fleet-wide

Same two checks as `04_site_resolution.ipynb` Section 7, run once across
every `CLEAN_SITE_IDS` site instead of a 50-site sample.


In [9]:
storage_site_ids = Signal.sites_with_storage_circuits(meta_local)
fleet_storage_site_ids = sorted(set(storage_site_ids) & set(RESOLUTION_SITE_IDS))
print(f"{len(fleet_storage_site_ids):,} site(s) in this run's cohort have an explicit, "
      "name-detected battery/EV circuit_type.")

circuit_polarity_lookup = meta_local[["circuit_id", "circuit_polarity"]].drop_duplicates("circuit_id")

fleet_interval_table = Resolution.build_interval_table(
    fleet_sample, fleet_resolution, nominal_interval_minutes=Config.SOURCE_INTERVAL_MINUTES,
)
reconstructed = Signal.reconstruct_gross_load(fleet_interval_table, circuit_polarity_lookup)
reconstructed["t_stamp"] = Plots.to_aest(reconstructed["t_stamp"])
reconstruction_report = Signal.evaluate_load_reconstruction(reconstructed)

flagged_site_ids = set(
    reconstruction_report.loc[reconstruction_report.likely_storage_or_sign_issue, "site_id"]
)
print(f"{len(flagged_site_ids):,} of {len(reconstruction_report):,} sites show a NIGHT-time "
      "negative reconstructed load on the sampled day.")


190 site(s) in this run's cohort have an explicit, name-detected battery/EV circuit_type.
130 of 8,837 sites show a NIGHT-time negative reconstructed load on the sampled day.


### 1e. Drop flagged AND manual-review sites entirely (most conservative choice)

Agreed 2026-08-27: a site failing the reconstruction/storage check, or
flagged `needs_manual_review` (an ambiguous same-type duplicate with no
established tie-break rule), is dropped entirely rather than guessed at.
Each exclusion keeps its OWN reason in the audit trail, so a future reader
can tell a storage/sign issue from an unresolved duplicate ambiguity.


In [10]:
manual_review_site_ids = sorted(
    fleet_resolution.loc[fleet_resolution.needs_manual_review, "site_id"].unique().tolist()
)
print(f"{len(manual_review_site_ids):,} site(s) flagged needs_manual_review -- "
      "dropped entirely (conservative choice, agreed 2026-08-27).")

fleet_resolution_final = Resolution.exclude_flagged_sites(
    fleet_resolution, set(fleet_storage_site_ids) | flagged_site_ids,
    reason="storage_or_sign_issue",
)
fleet_resolution_final = Resolution.exclude_flagged_sites(
    fleet_resolution_final, manual_review_site_ids,
    reason="manual_review_conservatively_dropped",
)

day_coverage_report = Resolution.build_coverage_report(fleet_resolution_final)
display(pd.DataFrame([day_coverage_report]))
print(f"\n{day_coverage_report['n_sites']:,} CLEAN_SITE_IDS sites scanned on the sampled day; "
      f"{int(fleet_resolution_final.groupby('site_id').kept.any().sum()):,} have at least one "
      "surviving circuit after every day-level check.")


2,754 site(s) flagged needs_manual_review -- dropped entirely (conservative choice, agreed 2026-08-27).


,n_sites,n_no_intervention,n_auto_resolved_duplicate_cross_type,n_auto_resolved_inactive,n_flagged_manual_review,n_excluded_storage_or_sign_issue,n_excluded_inactive_full_year,n_excluded_storage_or_sign_issue_full_year,n_power_correction_applied
0,14353,10823,1028,219,2754,316,0,0,2406



14,353 CLEAN_SITE_IDS sites scanned on the sampled day; 11,351 have at least one surviving circuit after every day-level check.


## 2. Extract: full year for the survivors only

Never for a site already excluded above -- extraction only spends Athena
scan and local disk on circuits the day-level resolution decided are worth
a full year of data.


In [11]:
'''
FULL_YEAR_MONTHS = Extract.months_in_range(*TRIAL_MONTHS)

surviving_circuits = fleet_resolution_final.loc[
    fleet_resolution_final.kept, ["circuit_id", "is_pv"]
]
print(f"Extracting {len(FULL_YEAR_MONTHS):,} months for {len(surviving_circuits):,} "
      "surviving circuits.")

extraction_manifest = Extract.run_extraction(
    Athena.aq, surviving_circuits, FULL_YEAR_MONTHS, Config.STORE_DIR,
    chunk_size=800,
)
print(f"Landed {extraction_manifest.n_rows.sum():,} rows across "
      f"{len(extraction_manifest):,} (month, is_pv, chunk) queries.")
Athena.scan_report()

'''

'\nFULL_YEAR_MONTHS = Extract.months_in_range(*TRIAL_MONTHS)\n\nsurviving_circuits = fleet_resolution_final.loc[\n    fleet_resolution_final.kept, ["circuit_id", "is_pv"]\n]\nprint(f"Extracting {len(FULL_YEAR_MONTHS):,} months for {len(surviving_circuits):,} "\n      "surviving circuits.")\n\nextraction_manifest = Extract.run_extraction(\n    Athena.aq, surviving_circuits, FULL_YEAR_MONTHS, Config.STORE_DIR,\n    chunk_size=800,\n)\nprint(f"Landed {extraction_manifest.n_rows.sum():,} rows across "\n      f"{len(extraction_manifest):,} (month, is_pv, chunk) queries.")\nAthena.scan_report()\n\n'

In [12]:
FULL_YEAR_MONTHS = Extract.months_in_range(*TRIAL_MONTHS)

surviving_circuits = fleet_resolution_final.loc[
    fleet_resolution_final.kept, ["circuit_id", "is_pv"]
]
print(f"Extracting {len(FULL_YEAR_MONTHS):,} months for {len(surviving_circuits):,} "
      "surviving circuits.")

# extraction_manifest = Extract.run_extraction(
#     Athena.aq, surviving_circuits, FULL_YEAR_MONTHS, Config.STORE_DIR,
#     chunk_size=800,
# )
# print(f"Landed {extraction_manifest.n_rows.sum():,} rows across "
#       f"{len(extraction_manifest):,} (month, is_pv, chunk) queries.")
# Athena.scan_report()
extraction_manifest = pd.read_csv(Config.ARTEFACT_DIR / "phase5_extraction_manifest.csv")  # reuse the prior run's manifest
print(f"Skipped re-extraction -- reused manifest with {extraction_manifest.n_rows.sum():,} rows already on disk.")

Extracting 12 months for 29,919 surviving circuits.
Skipped re-extraction -- reused manifest with 1,491,143,764 rows already on disk.


## 3. Revalidate against the full landed year

A circuit that goes inactive mid-year is dropped on its own; a site that
fails the reconstruction/storage check in any landed month is excluded
entirely -- see `lib/ami_revalidate.py`'s module docstring for why the two
are handled at different granularities.


In [13]:
inactive_over_history = Reval.revalidate_inactive_circuits_over_history(
    Reval.iter_month_partitions(Config.STORE_DIR, FULL_YEAR_MONTHS)
)
print(f"{len(inactive_over_history):,} circuit(s) newly inactive in at least one landed month "
      "(not caught by the single sampled day).")

reconstruction_over_history = Reval.revalidate_reconstruction_over_history(
    Reval.iter_month_partitions(Config.STORE_DIR, FULL_YEAR_MONTHS),
    fleet_resolution_final, circuit_polarity_lookup,
)
print(f"{len(reconstruction_over_history):,} site(s) newly failing the reconstruction check "
      "in at least one landed month.")

final_resolution = Reval.apply_full_year_findings(
    fleet_resolution_final, inactive_over_history, reconstruction_over_history,
)


327 circuit(s) newly inactive in at least one landed month (not caught by the single sampled day).
253 site(s) newly failing the reconstruction check in at least one landed month.


In [14]:
fleet_resolution_final[fleet_resolution_final["site_id"] == 360624571]

,site_id,circuit_id,circuit_type,device_id,is_pv,kept,drop_reason,needs_manual_review,power_correction_applied,implied_interval_minutes
33771,360624571,631836,ac_load_net,112534,False,True,None,False,True,4.895823
34577,360624571,631838,pv_site_net,112534,True,True,None,False,True,4.890260


In [15]:
flagged_by_site = (
    final_resolution[final_resolution.kept]
    .groupby("site_id")["power_correction_applied"]
    .any()
)
n_kept_sites = flagged_by_site.index.nunique()
n_flagged_sites = int(flagged_by_site.sum())
n_clean_sites = n_kept_sites - n_flagged_sites

print(f"{n_flagged_sites:,} of {n_kept_sites:,} kept sites ({n_flagged_sites / n_kept_sites:.1%}) "
      f"have at least one power_correction_applied=True circuit.")
print(f"{n_clean_sites:,} sites would remain if you dropped all of them.")

1,119 of 11,080 kept sites (10.1%) have at least one power_correction_applied=True circuit.
9,961 sites would remain if you dropped all of them.


## 4. Final tally


In [16]:
SITE_METADATA_COLUMNS = [
    "site_id", "state", "postcode", "dnsp_name", "ac_capacity_kw", "dc_capacity_kw",
    "export_limit_kw", "inverter_count", "voltage_class", "manufacturer", "model",
    "flex_export_detected", "monitoring_start", "pv_install_date",
]
site_level_meta = (
    meta_local[meta_local.site_id.isin(RESOLUTION_SITE_IDS)]
    [[c for c in SITE_METADATA_COLUMNS if c in meta_local.columns]]
    .drop_duplicates(subset="site_id")
)

site_metadata = Resolution.build_site_metadata(site_level_meta, final_resolution)
coverage_report = Resolution.build_coverage_report(final_resolution)
display(pd.DataFrame([coverage_report]))

n_final_sites = int(final_resolution.groupby("site_id").kept.any().sum())
print(f"\n{n_final_sites:,} of {coverage_report['n_sites']:,} sites in this run's cohort "
      "survive into the final synthetic AMI dataset.")


,n_sites,n_no_intervention,n_auto_resolved_duplicate_cross_type,n_auto_resolved_inactive,n_flagged_manual_review,n_excluded_storage_or_sign_issue,n_excluded_inactive_full_year,n_excluded_storage_or_sign_issue_full_year,n_power_correction_applied
0,14353,10386,1028,219,2754,316,235,253,2236



11,080 of 14,353 sites in this run's cohort survive into the final synthetic AMI dataset.


## 5. Build: `ami_raw` / `ami_meter`

One landed month at a time -- see `lib/ami_build.py`'s module docstring for
what each table contains and why `ami_meter` stays in Watts, per phase,
rather than pre-converting to kWh at a chosen interval (that decision --
`ami_config.TARGET_INTERVAL_MINUTES` -- is still explicitly unresolved).


In [17]:
site_capacity_lookup = Athena.aq(f"""
    SELECT site_id, max(S_99) AS S_99, max(ac_capacity_kw) AS ac_capacity_kw
    FROM meta_up23c
    WHERE circuit_id IN ({",".join(str(c) for c in surviving_circuits.circuit_id)})
    GROUP BY site_id
""")

In [18]:
flagged = Resolution.sites_with_power_correction(final_resolution)
clean_site_ids = flagged[~flagged].index
dropped_site_ids = flagged[flagged].index
print(f"{len(dropped_site_ids):,} of {flagged.index.nunique():,} kept sites "
      f"would be excluded ({len(dropped_site_ids) / flagged.index.nunique():.1%}).")

clean_resolution = final_resolution[final_resolution.site_id.isin(clean_site_ids)]

1,119 of 11,080 kept sites would be excluded (10.1%).


In [19]:
# clean_resolution = drop the CATCH meters that have the broken timestamps
RESOLUTION = clean_resolution

In [20]:
build_manifest = Build.run_build(
    Reval.iter_month_partitions(Config.STORE_DIR, FULL_YEAR_MONTHS),
    RESOLUTION, circuit_polarity_lookup,
    Config.store_path("ami_raw"), Config.store_path("ami_meter"),
    site_capacity=site_capacity_lookup,
    apply_power_correction=False,   # match structured_data's treatment
)
print(f"ami_raw: {build_manifest.n_raw_rows.sum():,} rows across "
      f"{len(build_manifest):,} months")
print(f"ami_meter: {build_manifest.n_meter_rows.sum():,} rows across "
      f"{len(build_manifest):,} months")
display(build_manifest)

ami_raw: 512,050,770 rows across 12 months
ami_meter: 656,973,671 rows across 12 months


,year,month,n_raw_rows,n_meter_rows,raw_path,meter_path
0,2025,1,52959308,72074016,C:\Users\z3553082\AppData\Local\ciccada\ami_st...,C:\Users\z3553082\AppData\Local\ciccada\ami_st...
1,2025,2,45674121,61179897,C:\Users\z3553082\AppData\Local\ciccada\ami_st...,C:\Users\z3553082\AppData\Local\ciccada\ami_st...
2,2025,3,46229253,60235823,C:\Users\z3553082\AppData\Local\ciccada\ami_st...,C:\Users\z3553082\AppData\Local\ciccada\ami_st...
3,2025,4,43971969,56437421,C:\Users\z3553082\AppData\Local\ciccada\ami_st...,C:\Users\z3553082\AppData\Local\ciccada\ami_st...
4,2025,5,43810020,55426700,C:\Users\z3553082\AppData\Local\ciccada\ami_st...,C:\Users\z3553082\AppData\Local\ciccada\ami_st...
5,2025,6,33253135,41316044,C:\Users\z3553082\AppData\Local\ciccada\ami_st...,C:\Users\z3553082\AppData\Local\ciccada\ami_st...
6,2025,7,43754599,54926722,C:\Users\z3553082\AppData\Local\ciccada\ami_st...,C:\Users\z3553082\AppData\Local\ciccada\ami_st...
7,2025,8,43019019,54100259,C:\Users\z3553082\AppData\Local\ciccada\ami_st...,C:\Users\z3553082\AppData\Local\ciccada\ami_st...
8,2025,9,40584282,51052884,C:\Users\z3553082\AppData\Local\ciccada\ami_st...,C:\Users\z3553082\AppData\Local\ciccada\ami_st...
9,2025,10,41018166,51729523,C:\Users\z3553082\AppData\Local\ciccada\ami_st...,C:\Users\z3553082\AppData\Local\ciccada\ami_st...


In [21]:
phase_split_manifest = Build.run_phase_split_build(
    Reval.iter_month_partitions(Config.STORE_DIR, FULL_YEAR_MONTHS),
    RESOLUTION, circuit_polarity_lookup,
    Config.store_path("ami_raw_phaseseparate"),
    apply_power_correction=False,   #
)
print(f"ami_raw_phaseseparate: {phase_split_manifest.n_rows.sum():,} rows across "
      f"{len(phase_split_manifest):,} months")
display(phase_split_manifest)

ami_raw_phaseseparate: 656,973,671 rows across 12 months


,year,month,n_rows,path
0,2025,1,72074016,C:\Users\z3553082\AppData\Local\ciccada\ami_st...
1,2025,2,61179897,C:\Users\z3553082\AppData\Local\ciccada\ami_st...
2,2025,3,60235823,C:\Users\z3553082\AppData\Local\ciccada\ami_st...
3,2025,4,56437421,C:\Users\z3553082\AppData\Local\ciccada\ami_st...
4,2025,5,55426700,C:\Users\z3553082\AppData\Local\ciccada\ami_st...
5,2025,6,41316044,C:\Users\z3553082\AppData\Local\ciccada\ami_st...
6,2025,7,54926722,C:\Users\z3553082\AppData\Local\ciccada\ami_st...
7,2025,8,54100259,C:\Users\z3553082\AppData\Local\ciccada\ami_st...
8,2025,9,51052884,C:\Users\z3553082\AppData\Local\ciccada\ami_st...
9,2025,10,51729523,C:\Users\z3553082\AppData\Local\ciccada\ami_st...


## 6. Save small, human-reviewable artefacts to `artefacts/`

The deliverable itself (`ami_raw`/`ami_meter`, potentially billions of
rows) lives in the local Parquet store above, NOT here -- these are the
small audit-trail tables, same convention as Phase 4.


In [22]:
ARTEFACT_DIR = Config.ARTEFACT_DIR
ARTEFACT_DIR.mkdir(parents=True, exist_ok=True)

site_metadata.to_csv(ARTEFACT_DIR / "phase5_site_metadata_full_fleet.csv", index=False)
pd.DataFrame([coverage_report]).to_csv(ARTEFACT_DIR / "phase5_coverage_report_full_fleet.csv", index=False)
extraction_manifest.to_csv(ARTEFACT_DIR / "phase5_extraction_manifest.csv", index=False)
build_manifest.to_csv(ARTEFACT_DIR / "phase5_build_manifest.csv", index=False)
phase_split_manifest.to_csv(ARTEFACT_DIR / "phase5_phase_split_manifest.csv", index=False)   # NEW
inactive_over_history.to_csv(ARTEFACT_DIR / "phase5_inactive_over_history.csv", index=False)
reconstruction_over_history.to_csv(ARTEFACT_DIR / "phase5_reconstruction_over_history.csv", index=False)

print(f"Saved site metadata, coverage report, and provenance manifests to {ARTEFACT_DIR}")

Saved site metadata, coverage report, and provenance manifests to C:\Users\z3553082\OneDrive - UNSW\Documents\GitHub\CICCADA\bms_sa_review\ami_data_analysis\artefacts
